# Volume 12, Chapter 2: Ideals and the splitting of primes (Python companion)

Exercises first, solutions at the end. We factor primes in number fields with PARI/GP (through `cypari2`), test the theorems of the chapter on many fields, watch Frobenius statistics converge to Chebotarev's prediction, and implement from scratch the algorithm that makes the Kummer–Dedekind theorem practical: Cantor and Zassenhaus' factorisation of polynomials over $\mathbb F_p$. Requires `cypari2`, `sympy` and `numpy`.

In [ ]:
import random, time
from collections import Counter
import cypari2
pari = cypari2.Pari()
pari.allocatemem(4 * 10**8, silent=True)

def decomposition(nf, p):
    # The list of (e, f) for the primes of nf above p.
    return sorted((int(P.pr_get_e()), int(P.pr_get_f())) for P in pari.idealprimedec(nf, p))

## Exercises

**P2.1 (★) Splitting in $\mathbb Q(\sqrt[3]2)$.** For all primes $p<10^5$, compute the decomposition type of $p$ (the list of pairs $(e_i,f_i)$) in $K=\mathbb Q(\sqrt[3]2)$. Check $\sum e_if_i=3$ (Theorem 2.11), check the rule of Exercise 2.2, and compute the proportion of primes of each type. The Galois closure has group $S_3$; compare with the proportions of the identity, the transpositions and the $3$-cycles in $S_3$, using Proposition 2.22.

**P2.2 (★) Ramification is the discriminant.** For $300$ random number fields of degrees $2$ to $5$, check that the ramified primes (some $e_i\ge2$) are exactly the prime divisors of $d_K$ (Theorem 2.15). Then check Stickelberger's congruence for odd unramified primes (Exercise 2.11): $\left(\frac{d_K}p\right)=(-1)^{n-g}$ for $p<100$.

**P2.3 (★★) Norms of ideals.** In $K=\mathbb Q(\sqrt[3]{10})$ take random ideals $I=(a,b)$ generated by two random integers of $K$. Check $N(I)=[\mathcal O_K:I]$ by computing the index of the lattice $I$ (the determinant of its Hermite normal form in an integral basis) and comparing with `idealnorm`. Check multiplicativity on products.

**P2.4 (★★) Guessing Galois groups.** For $f_1=X^5-X-1$ and $f_2=X^5-5X+12$, record the degrees of the irreducible factors of $f$ modulo $p$ for the primes $p<10^5$ not dividing $\operatorname{disc}(f)$, and tabulate the frequencies. Which cycle types never occur for $f_2$? Find the smallest group of permutations of $5$ points compatible with the statistics, and compare with PARI's `polgalois`.

**P2.5 (★★★) CS application: Cantor–Zassenhaus.** Implement, with polynomials over $\mathbb F_p$ ($p$ odd) as lists of coefficients:

1. *Squarefree check and distinct-degree factorisation:* for $d=1,2,\dots$, $g_d=\gcd\bigl(X^{p^d}-X,\ f\bigr)$ collects the irreducible factors of degree $d$ (Volume 2, Theorem 9.9: $X^{p^d}-X$ is the product of the monic irreducible polynomials of degree dividing $d$).
2. *Equal-degree splitting:* if $g$ is a product of $r\ge2$ irreducible factors of degree $d$, then for a random $a$ of degree $<\deg g$, $\gcd\bigl(a^{(p^d-1)/2}-1,\ g\bigr)$ is a proper factor with probability about $1-2^{1-r}$. Explain why, with the Chinese remainder theorem $\mathbb F_p[X]/(g)\cong\mathbb F_{p^d}^r$.

Check your factorisations against `sympy`, then use them to compute the splitting of $p$ in $\mathbb Q(\zeta_{m})$ for $m=7,11,13$ via Kummer–Dedekind, and confirm Theorem 2.19 ($f$ is the order of $p$ modulo $m$). Finally time the algorithm on random polynomials of degree $16$ to $96$ modulo a $20$-digit prime.

## Solutions

### P2.1
**How to find it.** `idealprimedec` returns the primes above $p$ with their $e$ and $f$. Proposition 2.22 says that for unramified $p$ a Frobenius acts on the three embeddings with cycle type $(f_1,\dots,f_g)$. In $S_3$, the identity (type $1,1,1$) has proportion $\frac16$, the three transpositions (type $1,2$) $\frac12$, and the two $3$-cycles (type $3$) $\frac13$.

In [ ]:
K = pari.nfinit('x^3 - 2')
types = Counter()
for p in pari.primes(9592):          # the primes below 10^5
    p = int(p)
    t = decomposition(K, p)
    assert sum(e * f for e, f in t) == 3
    if p >= 5:
        if p % 3 == 2:
            assert t == [(1, 1), (1, 2)]
        else:
            is_cube = pow(2, (p - 1) // 3, p) == 1
            assert t == ([(1, 1)] * 3 if is_cube else [(1, 3)])
    types[tuple(t)] += 1
total = sum(types.values())
for t, c in types.most_common():
    print(t, c, round(c / total, 4))

The rule of Exercise 2.2 holds for every $p<10^5$ (by Euler's criterion in the cyclic group $\mathbb F_p^*$, $2$ is a cube iff $2^{(p-1)/3}\equiv1$ when $p\equiv1\bmod 3$). Only $2$ and $3$ ramify, each as $P^3$. The proportions are close to $\frac12$, $\frac13$, $\frac16$: the primes $p\equiv2\bmod3$ are half of all primes (Dirichlet), and among $p\equiv1\bmod3$, $2$ is a cube for one third of them. Chapter 8 proves these densities (Chebotarev).

### P2.2
**How to find it.** Random monic polynomials, keep the irreducible ones, then compare two sets of primes. For the congruence, $g$ is the number of primes above $p$.

In [ ]:
random.seed(5)
fields = 0
while fields < 300:
    n = random.randint(2, 5)
    f = pari('Pol(%s)' % ([1] + [random.randint(-12, 12) for _ in range(n)]))
    if not pari.polisirreducible(f):
        continue
    fields += 1
    nf = pari.nfinit(f)
    d = int(pari.nf_get_disc(nf)) if hasattr(pari, 'nf_get_disc') else int(nf[2])
    div = {int(q) for q in pari.factor(abs(d))[0]} if abs(d) > 1 else set()
    ram = set()
    for q in div | {2, 3, 5, 7}:
        if any(e > 1 for e, _ in decomposition(nf, q)):
            ram.add(q)
    assert ram == div, (f, d)
    for p in pari.primes(25):
        p = int(p)
        if p == 2 or d % p == 0:
            continue
        g = len(decomposition(nf, p))
        assert int(pari.kronecker(d, p)) == (-1) ** (n - g)
print(fields, "fields: ramified primes = primes dividing d_K, and Stickelberger's congruence holds")

Both theorems hold on every example. (We tested ramification at the prime divisors of $d_K$ and at $2,3,5,7$; Theorem 2.15 says no other prime can ramify, which the second half of the test supports indirectly: an unramified prime always gives the right sign.)

### P2.3
**How to find it.** PARI stores an ideal as a matrix in Hermite normal form whose columns are a $\mathbb Z$-basis of $I$ in terms of the integral basis; its determinant is $[\mathcal O_K:I]$ (Volume 2, Corollary 7.8).

In [ ]:
K = pari.nfinit('x^3 - 10')
random.seed(6)
for _ in range(5):
    a = pari('[%d, %d, %d]~' % tuple(random.randint(-20, 20) for _ in range(3)))
    b = pari('[%d, %d, %d]~' % tuple(random.randint(-20, 20) for _ in range(3)))
    I = pari.idealadd(K, a, b)
    J = pari.idealadd(K, b, 7)
    index = abs(int(pari.matdet(pari.idealhnf(K, I))))
    print("N(I) =", pari.idealnorm(K, I), " index =", index,
          " multiplicative:", pari.idealnorm(K, pari.idealmul(K, I, J)) == pari.idealnorm(K, I) * pari.idealnorm(K, J))

### P2.4
**How to find it.** Count factor-degree patterns. A cycle type that never occurs among tens of thousands of primes is (by Chebotarev) absent from the group.

In [ ]:
def cycle_stats(fstr, bound=10**5):
    f = pari(fstr)
    D = int(pari.poldisc(f))
    stats = Counter()
    for p in pari.primes([3, bound]):
        p = int(p)
        if D % p == 0:
            continue
        degs = tuple(sorted(int(d) for d in pari.factormod(f, p, 1)[0]))
        stats[degs] += 1
    tot = sum(stats.values())
    return {k: round(v / tot, 4) for k, v in sorted(stats.items())}

print("X^5-X-1  :", cycle_stats('x^5 - x - 1'))
print("X^5-5X+12:", cycle_stats('x^5 - 5*x + 12'))
print(pari.polgalois('x^5 - x - 1'), pari.polgalois('x^5 - 5*x + 12'))

For $X^5-X-1$ all seven cycle types of $S_5$ occur with the frequencies of Figure 2.2. For $X^5-5X+12$ only three types occur: $1^5$ (about $\frac1{10}$), $1\,2^2$ (about $\frac12$) and $5$ (about $\frac25$). A group of order $N$ in which the identity has proportion $\frac1{10}$ has $N=10$; with four $5$-cycles and five double transpositions, it is the dihedral group $D_5$ of the regular pentagon, acting on its vertices. PARI agrees: order $10$. (The discriminant $2^{12}5^6$ is a square, consistent with $D_5\subseteq A_5$.)

### P2.5
**How to find it.** In $R=\mathbb F_p[X]/(g)\cong\mathbb F_{p^d}^r$, the map $a\mapsto a^{(p^d-1)/2}$ acts coordinatewise, and in each coordinate it sends a non-zero element to $\pm1$, each sign for half of $\mathbb F_{p^d}^*$ (the quadratic character of a cyclic group of even order). So $a^{(p^d-1)/2}-1$ vanishes in the coordinates where the sign is $+1$ and not in the others, and its gcd with $g$ is the product of the corresponding factors. It is a proper factor unless all signs agree, which has probability about $2\cdot2^{-r}$.

In [ ]:
# Polynomials over F_p as lists of coefficients, constant term first, no trailing zeros.
def trim(a):
    while a and a[-1] == 0:
        a.pop()
    return a

def sub(a, b, p):
    n = max(len(a), len(b))
    return trim([((a[i] if i < len(a) else 0) - (b[i] if i < len(b) else 0)) % p for i in range(n)])

def mul(a, b, p):
    if not a or not b:
        return []
    r = [0] * (len(a) + len(b) - 1)
    for i, x in enumerate(a):
        if x:
            for j, y in enumerate(b):
                r[i + j] = (r[i + j] + x * y) % p
    return trim(r)

def divmod_poly(a, b, p):
    a = a[:]
    inv = pow(b[-1], p - 2, p)
    q = [0] * max(len(a) - len(b) + 1, 0)
    while len(a) >= len(b):
        c = a[-1] * inv % p
        k = len(a) - len(b)
        q[k] = c
        for i, y in enumerate(b):
            a[i + k] = (a[i + k] - c * y) % p
        trim(a)
    return trim(q), a

def monic(a, p):
    inv = pow(a[-1], p - 2, p)
    return [x * inv % p for x in a]

def gcd(a, b, p):
    while b:
        a, b = b, divmod_poly(a, b, p)[1]
    return monic(a, p) if a else a

def powmod(a, e, f, p):
    r, a = [1], divmod_poly(a, f, p)[1]
    while e:
        if e & 1:
            r = divmod_poly(mul(r, a, p), f, p)[1]
        a = divmod_poly(mul(a, a, p), f, p)[1]
        e >>= 1
    return r

def derivative(a, p):
    return trim([i * a[i] % p for i in range(1, len(a))])

def distinct_degree(f, p):
    # f monic squarefree; returns [(d, product of the irreducible factors of degree d)].
    out, h, d = [], [0, 1], 0
    while len(f) - 1 >= 2 * (d + 1):
        d += 1
        h = powmod(h, p, f, p)                 # X^(p^d) mod f
        g = gcd(f, sub(h, [0, 1], p), p)
        if len(g) > 1:
            out.append((d, g))
            f = divmod_poly(f, g, p)[0]
            h = divmod_poly(h, f, p)[1]
    if len(f) > 1:
        out.append((len(f) - 1, f))
    return out

def equal_degree(g, d, p):
    if len(g) - 1 == d:
        return [g]
    while True:
        a = trim([random.randrange(p) for _ in range(len(g) - 1)])
        if len(a) < 2:
            continue
        b = sub(powmod(a, (p**d - 1) // 2, g, p), [1], p)
        h = gcd(g, b, p)
        if 1 < len(h) < len(g):
            return equal_degree(h, d, p) + equal_degree(divmod_poly(g, h, p)[0], d, p)

def factor_squarefree(f, p):
    f = monic(f, p)
    assert len(gcd(f, derivative(f, p), p)) == 1, "not squarefree"
    return sorted((fac for d, g in distinct_degree(f, p) for fac in equal_degree(g, d, p)),
                  key=lambda q: (len(q), q))

In [ ]:
from sympy import Poly, symbols
x = symbols('x')
random.seed(7)
p = 1009
agree = 0
for _ in range(50):
    f = [random.randrange(p) for _ in range(12)] + [1]
    if len(gcd(f, derivative(f, p), p)) > 1:
        continue
    ours = sorted(len(q) - 1 for q in factor_squarefree(f, p))
    theirs = sorted(fac.degree() for fac, e in Poly(list(reversed(f)), x, modulus=p).factor_list()[1])
    agree += ours == theirs
print("agreement with sympy:", agree)

def cyclotomic(m):
    return [int(c) for c in reversed(pari.Vec(pari.polcyclo(m)))]

for m in (7, 11, 13):
    for p in (2, 3, 5, 17, 29, 47, 53):
        if p == 2 or m % p == 0:
            continue                           # our splitting step needs p odd
        degs = {len(q) - 1 for q in factor_squarefree(cyclotomic(m), p)}
        order = next(k for k in range(1, m) if pow(p, k, m) == 1)
        assert degs == {order}
print("Theorem 2.19 confirmed: every factor of Phi_m mod p has degree ord_m(p)")

In [ ]:
q = int(pari.nextprime(10**19))
for n in (16, 32, 64, 96):
    random.seed(n)
    f = [random.randrange(q) for _ in range(n)] + [1]
    t = time.perf_counter()
    facs = factor_squarefree(f, q)
    dt = time.perf_counter() - t
    print("degree %3d: %2d factors, degrees %s, %.2f s" % (n, len(facs), sorted(len(g) - 1 for g in facs), dt))

Our factorisations agree with `sympy`, and the factor degrees of $\Phi_m$ modulo $p$ are exactly the orders of $p$ modulo $m$, as Theorem 2.19 predicts. A random polynomial of degree $n$ has about $\log n$ irreducible factors (the same statistics as the cycles of a random permutation, by Chebotarev for the "generic" Galois group $S_n$; Volume 3 studies the cycles of random permutations).

Each step costs $O(n^2)$ operations per multiplication with schoolbook arithmetic, and $X^{p^d}$ needs $O(\log p)$ multiplications for each of up to $n/2$ values of $d$: about $O(n^3\log p)$ in total, hence the roughly cubic growth of the timings. With fast multiplication (FFT, Volume 4) and the baby-step giant-step variant of von zur Gathen and Shoup, the best algorithms (Kedlaya and Umans, 2008) run in time about $n^{1.5}\log p$. This polynomial-time factorisation is what makes Kummer–Dedekind (and the computation of $\mathcal O_K$ away from the hard primes, Chapter 1's notebook) efficient. Over $\mathbb Z$, by contrast, factoring integers has no known polynomial algorithm.